# DM2026 Lab 2 – Transformer model (run on Kaggle GPU)

Fine-tunes `roberta-large` with 5-fold cross-validation, averages the five fold models' test predictions,
and blends them with a TF-IDF + logistic regression model (blend weight chosen on out-of-fold predictions).

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings → **Internet on** (needed to download the model).
Then *Run All* (about 30-40 minutes). Output: `/kaggle/working/submission.csv`.

## 1. Setup and configuration

In [ ]:
import os, glob, json, random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# All tunable settings in one place.
CFG = dict(
    # roberta-base reached 0.5985 out-of-fold macro-F1; the large model (355M params) usually adds a few points.
    model_name="roberta-large",
    max_len=128,        # median post is ~68 characters, so 128 tokens covers almost everything
    batch_size=16,      # smaller batches so the large model fits in T4 memory
    epochs=4,
    lr=1e-5,            # large models are less stable, so a lower learning rate than base (2e-5)
    weight_decay=0.01,
    warmup_ratio=0.1,
    n_folds=5,
    seed=42,
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

def seed_everything(seed):
    # Fix every random source so a rerun gives the same folds and (nearly) the same scores.
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])

## 2. Preprocessing

In [ ]:
# Find the competition files whether they sit in /kaggle/input (Kaggle) or data/kaggle (local repo).
hits = glob.glob("/kaggle/input/**/final_posts.json", recursive=True) or glob.glob("data/kaggle/final_posts.json")
DATA_DIR = os.path.dirname(hits[0])
print("data:", DATA_DIR)

with open(f"{DATA_DIR}/final_posts.json", encoding="utf-8") as f:
    posts = pd.DataFrame(json.load(f))                       # post_id, text
split = pd.read_csv(f"{DATA_DIR}/data_identification.csv")    # post_id, set
labels = pd.read_csv(f"{DATA_DIR}/emotion.csv")               # post_id, label

df = posts.merge(split, on="post_id")
train_df = df[df["set"] == "train"].merge(labels, on="post_id").reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)

# Minimal cleaning: the transformer's own tokenizer handles case, punctuation and emoji,
# so we only fill missing text and trim whitespace.
for part in (train_df, test_df):
    part["text"] = part["text"].fillna("").astype(str).str.strip()

# Encode the six emotion names as integers 0..5 for the classifier.
CLASSES = sorted(train_df["label"].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
train_df["y"] = train_df["label"].map(label2id)
print(train_df.shape, test_df.shape, CLASSES)

## 3. Feature engineering (tokenization)

In [ ]:
# Instead of hand-made features, the pretrained RoBERTa tokenizer turns each post into
# sub-word token ids; the model learns contextual features from them during fine-tuning.
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class PostDataset(Dataset):
    def __init__(self, texts, ys=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=CFG["max_len"])
        self.ys = None if ys is None else list(ys)
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.ys is not None:
            item["labels"] = self.ys[i]
        return item

def collate(batch):
    # Pad each batch only to its own longest post (faster than padding everything to max_len).
    labels = [b.pop("labels") for b in batch] if "labels" in batch[0] else None
    out = tokenizer.pad(batch, return_tensors="pt")
    if labels is not None:
        out["labels"] = torch.tensor(labels)
    return out

lens = [len(x) for x in tokenizer(list(train_df["text"]))["input_ids"]]
print("token length: median", int(np.median(lens)), " 99th pct", int(np.percentile(lens, 99)))

## 4. Model: 5-fold fine-tuning

In [ ]:
def predict(model, loader):
    # Return raw logits for every post in the loader.
    model.eval(); out = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                out.append(model(**batch).logits.float().cpu())
    return torch.cat(out).numpy()

def train_fold(tr, va):
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=len(CLASSES)).to(DEVICE)
    tr_dl = DataLoader(PostDataset(tr["text"], tr["y"]), batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate)
    va_dl = DataLoader(PostDataset(va["text"], va["y"]), batch_size=64, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = len(tr_dl) * CFG["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_ratio"] * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

    best_f1, best_state, best_logits = -1, None, None
    for epoch in range(CFG["epochs"]):
        model.train()
        for batch in tr_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = model(**batch).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        logits = predict(model, va_dl)
        f1 = f1_score(va["y"], logits.argmax(1), average="macro")
        print(f"  epoch {epoch + 1}: val macro-F1 {f1:.4f}")
        # Keep the weights from the best epoch on this fold's validation set.
        if f1 > best_f1:
            best_f1, best_logits = f1, logits
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_f1, best_logits

skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
oof = np.zeros((len(train_df), len(CLASSES)))          # out-of-fold logits for an honest CV score
test_logits = np.zeros((len(test_df), len(CLASSES)))   # summed test logits across folds
test_dl = DataLoader(PostDataset(test_df["text"]), batch_size=64, collate_fn=collate)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["y"])):
    t0 = time.time()
    print(f"fold {fold + 1}/{CFG['n_folds']}")
    model, f1, va_logits = train_fold(train_df.iloc[tr_idx], train_df.iloc[va_idx])
    if f1 < 0.35:
        # Large models occasionally fail to start learning and predict one class for everything.
        # If that happens, retrain this fold once with a different random seed.
        print("  fold did not learn, retrying with another seed")
        del model; torch.cuda.empty_cache(); seed_everything(CFG["seed"] + 100 + fold)
        model, f1, va_logits = train_fold(train_df.iloc[tr_idx], train_df.iloc[va_idx])
    oof[va_idx] = va_logits
    test_logits += predict(model, test_dl) / CFG["n_folds"]
    print(f"  best {f1:.4f}  ({time.time() - t0:.0f}s)")
    del model; torch.cuda.empty_cache()

cv_f1 = f1_score(train_df["y"], oof.argmax(1), average="macro")
print(f"\nOverall out-of-fold macro-F1: {cv_f1:.4f}")

## 5. Blend with a TF-IDF model

In [ ]:
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

# The TF-IDF baseline and RoBERTa-base disagreed on ~40% of test posts, so they make different
# mistakes; averaging their probabilities can beat either one alone.
def tfidf_model():
    feats = make_union(
        TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
    )
    return make_pipeline(feats, LogisticRegression(C=4, max_iter=3000, class_weight="balanced"))

# Out-of-fold TF-IDF probabilities on the SAME folds as the transformer, so the blend weight is chosen fairly.
oof_tfidf = np.zeros_like(oof)
for tr_idx, va_idx in skf.split(train_df, train_df["y"]):
    m = tfidf_model().fit(train_df["text"].iloc[tr_idx], train_df["y"].iloc[tr_idx])
    oof_tfidf[va_idx] = m.predict_proba(train_df["text"].iloc[va_idx])
test_tfidf = tfidf_model().fit(train_df["text"], train_df["y"]).predict_proba(test_df["text"])

# Try blend weights w (share of the transformer) and keep the one with the best out-of-fold macro-F1.
oof_tr, test_tr = softmax(oof, axis=1), softmax(test_logits, axis=1)
results = {}
for w in np.arange(0.0, 1.01, 0.1):
    results[round(w, 1)] = f1_score(train_df["y"], (w * oof_tr + (1 - w) * oof_tfidf).argmax(1), average="macro")
for w, f in results.items():
    print(f"  weight on transformer {w:.1f}: OOF macro-F1 {f:.4f}")
best_w = max(results, key=results.get)
print(f"best weight {best_w} -> {results[best_w]:.4f}  (transformer alone {results[1.0]:.4f}, TF-IDF alone {results[0.0]:.4f})")
test_probs = best_w * test_tr + (1 - best_w) * test_tfidf

## 6. Submission

In [ ]:
# Blended probabilities -> most likely emotion for each test post.
pred = [CLASSES[i] for i in test_probs.argmax(1)]
submission = pd.DataFrame({"post_id": test_df["post_id"], "label": pred})

# Match the sample file's row order and size exactly.
sample = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")
submission = sample[["post_id"]].merge(submission, on="post_id", how="left")
assert submission["label"].notna().all() and len(submission) == len(sample)

# Kaggle expects the ID column to be named "id" (the sample file says post_id, but the grader rejects it).
submission = submission.rename(columns={"post_id": "id"})
submission.to_csv("submission.csv", index=False)

# Also save the transformer-only predictions, as a second option to submit.
only_tr = sample[["post_id"]].merge(
    pd.DataFrame({"post_id": test_df["post_id"], "label": [CLASSES[i] for i in test_logits.argmax(1)]}), on="post_id")
only_tr.rename(columns={"post_id": "id"}).to_csv("submission_transformer_only.csv", index=False)
np.save("test_logits_roberta.npy", test_logits)
np.save("oof_logits_roberta.npy", oof)
print(submission["label"].value_counts())